In [1]:
import sys
!{sys.executable} -m pip install python-dotenv pandas requests sqlalchemy psycopg2-binary openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

BLS_KEY = os.getenv("BLS_API_KEY")
FRED_KEY = os.getenv("FRED_API_KEY")

print("BLS key loaded:", BLS_KEY is not None)
print("FRED key loaded:", FRED_KEY is not None)

BLS key loaded: True
FRED key loaded: True


In [3]:
import requests

# LNS14000000 is the BLS series ID for national unemployment rate
bls_url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
bls_payload = {
    "seriesid": ["LNS14000000"],
    "startyear": "2022",
    "endyear": "2024",
    "registrationkey": BLS_KEY
}

bls_response = requests.post(bls_url, json=bls_payload)
bls_data = bls_response.json()

print("BLS status:", bls_data["status"])
print("First few records:", bls_data["Results"]["series"][0]["data"][:3])

BLS status: REQUEST_SUCCEEDED
First few records: [{'year': '2024', 'period': 'M12', 'periodName': 'December', 'value': '4.1', 'footnotes': [{}]}, {'year': '2024', 'period': 'M11', 'periodName': 'November', 'value': '4.2', 'footnotes': [{}]}, {'year': '2024', 'period': 'M10', 'periodName': 'October', 'value': '4.1', 'footnotes': [{}]}]


In [5]:
fred_url = "https://api.stlouisfed.org/fred/series/observations"
fred_params = {
    "series_id": "UNRATE",
    "api_key": FRED_KEY,
    "file_type": "json",
    "observation_start": "2022-01-01",
    "observation_end": "2024-12-31"
}

fred_response = requests.get(fred_url, params=fred_params)
fred_data = fred_response.json()

print("FRED observation count:", len(fred_data["observations"]))
print("First record:", fred_data["observations"][0])

FRED observation count: 36
First record: {'realtime_start': '2026-04-02', 'realtime_end': '2026-04-02', 'date': '2022-01-01', 'value': '4.0'}


In [4]:
import pandas as pd

oews = pd.read_excel("../data/raw/oews_national.xlsx")

print(oews.shape)
print(oews.columns.tolist())
print(oews.head(3))

(1403, 32)
['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']
   AREA AREA_TITLE  AREA_TYPE PRIM_STATE  NAICS     NAICS_TITLE  \
0    99       U.S.          1         US      0  Cross-industry   
1    99       U.S.          1         US      0  Cross-industry   
2    99       U.S.          1         US      0  Cross-industry   

          I_GROUP  OWN_CODE OCC_CODE               OCC_TITLE  ... H_MEDIAN  \
0  cross-industry      1235  00-0000         All Occupations  ...     23.8   
1  cross-industry      1235  11-0000  Management Occupations  ...     58.7   
2  cross-industry      1235  11-1000          Top Executives  ...    50.48   

   H_PCT75  H_PCT90  A_PCT10  A_P